# Validación del entorno para el modelo MILP de Keccak

Este notebook verifica que el entorno de trabajo se encuentre correctamente configurado antes de implementar el modelo completo de Keccak.

Las validaciones incluyen:

- detección de la ruta principal del proyecto;
- importación de las librerías necesarias;
- reconocimiento del solver CBC mediante PuLP;
- comprobación de las dimensiones de Keccak para \(z=4\) y \(z=8\);
- resolución de un problema MILP binario sencillo;
- generación de la matriz inicial de experimentos.

En esta etapa todavía no se implementan las transformaciones $(\theta)$, $(\rho)$, $(\pi)$ y $(\chi)$.

## 1. Configuración de las rutas del proyecto

El código fuente del proyecto se encuentra dentro de la carpeta `src`.

Como el notebook se ejecuta desde la carpeta `notebooks`, se agrega manualmente la ruta `src` al entorno de Python para poder importar el paquete `keccak_milp`.

In [1]:
# ============================================================
# CONFIGURACIÓN DE RUTAS DEL PROYECTO
# ============================================================

from pathlib import Path
import sys


# Se asume que el notebook se ejecuta desde:
# PracticaCalificada/notebooks/
CURRENT_DIR = Path.cwd()

# Si estamos dentro de notebooks, la raíz es la carpeta superior.
if CURRENT_DIR.name == "notebooks":
    PROJECT_ROOT = CURRENT_DIR.parent
else:
    PROJECT_ROOT = CURRENT_DIR

SRC_DIR = PROJECT_ROOT / "src"

# Agregar src al path de Python solo si aún no está presente.
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))


print("=" * 70)
print("RUTAS DEL PROYECTO")
print("=" * 70)
print(f"Directorio actual : {CURRENT_DIR}")
print(f"Raíz del proyecto : {PROJECT_ROOT}")
print(f"Carpeta src       : {SRC_DIR}")
print(f"src existe        : {SRC_DIR.exists()}")
print("=" * 70)

RUTAS DEL PROYECTO
Directorio actual : d:\Documentos\000. MSC\3er Ciclo\Cripto\PracticaCalificada\notebooks
Raíz del proyecto : d:\Documentos\000. MSC\3er Ciclo\Cripto\PracticaCalificada
Carpeta src       : d:\Documentos\000. MSC\3er Ciclo\Cripto\PracticaCalificada\src
src existe        : True


## 2. Importación de dependencias

Se importan las librerías principales del proyecto:

- `NumPy`: operaciones numéricas;
- `Pandas`: organización de resultados;
- `Matplotlib`: generación posterior de gráficos;
- `PuLP`: construcción y resolución del modelo MILP;
- `keccak_milp`: paquete local del proyecto.

También se muestran las versiones instaladas y los solvers reconocidos por PuLP.

In [ ]:
# ============================================================
# IMPORTACIÓN DE LIBRERÍAS
# ============================================================

import platform

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import pulp

from keccak_milp.config import ExperimentConfig
from keccak_milp.solver import build_solver


print("=" * 70)
print("INFORMACIÓN DEL ENTORNO")
print("=" * 70)
print(f"Python      : {sys.version.split()[0]}")
print(f"Sistema     : {platform.platform()}")
print(f"NumPy       : {np.__version__}")
print(f"Pandas      : {pd.__version__}")
print(f"Matplotlib  : {matplotlib.__version__}")
print(f"PuLP        : {pulp.__version__}")
print("-" * 70)
print(f"Solvers disponibles: {available_solvers()}")
print("=" * 70)

INFORMACIÓN DEL ENTORNO
Python      : 3.12.10
Sistema     : Windows-10-10.0.19045-SP0
NumPy       : 2.5.1
Pandas      : 3.0.3
Matplotlib  : 3.11.0
PuLP        : 3.3.2
----------------------------------------------------------------------
Solvers disponibles: ['PULP_CBC_CMD']


In [ ]:
configuraciones = [
    ExperimentConfig(z=4, rounds=1),
    ExperimentConfig(z=4, rounds=2),
    ExperimentConfig(z=4, rounds=3),
    ExperimentConfig(z=8, rounds=1),
    ExperimentConfig(z=8, rounds=2),
    ExperimentConfig(z=8, rounds=3),
]

datos = []

for config in configuraciones:
    datos.append(
        {
            "z": config.z,
            "rondas": config.rounds,
            "bits_estado": config.state_bits,
            "sboxes_por_ronda": config.sboxes_per_round,
            "sboxes_disponibles": config.total_sboxes,
        }
    )

df_configuraciones = pd.DataFrame(datos)
df_configuraciones

## 3. Validación del solver CBC

Antes de implementar Keccak, se resolverá un problema MILP binario sencillo.

Se considera el siguiente modelo:

$[
\min z = x+y
]$

sujeto a:

$[
x+2y\geq 1
]$

$[
x,y\in\{0,1\}
]$

La solución óptima esperada es:

$[
x=0,\qquad y=1,\qquad z=1.
]$

Esta prueba confirma que PuLP puede comunicarse correctamente con CBC.

In [3]:
# ============================================================
# PRUEBA BÁSICA DEL SOLVER CBC
# ============================================================

# Crear un problema de minimización.
problema_prueba = pulp.LpProblem(
    name="validacion_cbc",
    sense=pulp.LpMinimize,
)

# Variables binarias.
x = pulp.LpVariable("x", lowBound=0, upBound=1, cat=pulp.LpBinary)
y = pulp.LpVariable("y", lowBound=0, upBound=1, cat=pulp.LpBinary)

# Función objetivo.
problema_prueba += x + y, "minimizar_variables_activas"

# Restricción.
problema_prueba += x + 2 * y >= 1, "restriccion_prueba"

# Configuración controlada del solver.
solver_cbc = pulp.COIN_CMD(
    msg=False,
    timeLimit=60,
    gapRel=0.0,
)

# Resolver.
problema_prueba.solve(solver_cbc)

# Recuperar resultados.
estado_prueba = pulp.LpStatus[problema_prueba.status]
objetivo_prueba = pulp.value(problema_prueba.objective)


print("=" * 70)
print("RESULTADO DE LA PRUEBA CBC")
print("=" * 70)
print(f"Estado   : {estado_prueba}")
print(f"x        : {x.value()}")
print(f"y        : {y.value()}")
print(f"Objetivo : {objetivo_prueba}")
print("=" * 70)


# Validaciones automáticas.
assert estado_prueba == "Optimal", (
    f"El solver no encontró una solución óptima. Estado: {estado_prueba}"
)

assert objetivo_prueba == 1.0, (
    f"El objetivo esperado era 1.0 y se obtuvo {objetivo_prueba}"
)

print("La validación del solver CBC fue completada correctamente.")

RESULTADO DE LA PRUEBA CBC
Estado   : Optimal
x        : 1.0
y        : 0.0
Objetivo : 1.0
La validación del solver CBC fue completada correctamente.


## 4. Configuraciones experimentales de Keccak

Keccak organiza su estado como una matriz de:

$[
5\times5
]$

palabras o *lanes*, donde cada palabra contiene $(z)$ bits. Por tanto, el tamaño del estado es:

$[
b=25z.
]$

Para este trabajo se consideran dos tamaños:

### Caso $(z=4)$

$[
b=25(4)=100\text{ bits}
]$

La capa $(\chi)$ procesa una S-box de 5 bits por cada fila y posición dentro de la palabra. Por tanto:

$[
5z=5(4)=20
]$

S-boxes disponibles por ronda.

### Caso $(z=8)$

$[
b=25(8)=200\text{ bits}
]$

El número de S-boxes disponibles por ronda es:

$[
5z=5(8)=40.
]$

Los experimentos se ejecutarán para 1, 2 y 3 rondas.

In [4]:
# ============================================================
# CREACIÓN DE LAS CONFIGURACIONES EXPERIMENTALES
# ============================================================

configuraciones = [
    ExperimentConfig(
        z=z,
        rounds=rondas,
        solver="cbc",
        time_limit_seconds=300,
        mip_gap=0.0,
        verbose=False,
    )
    for z in (4, 8)
    for rondas in (1, 2, 3)
]


for configuracion in configuraciones:
    print(
        f"z={configuracion.z}, "
        f"rondas={configuracion.rounds}, "
        f"estado={configuracion.state_bits} bits, "
        f"S-boxes/ronda={configuracion.sboxes_per_round}, "
        f"S-boxes disponibles={configuracion.total_sboxes}"
    )

z=4, rondas=1, estado=100 bits, S-boxes/ronda=20, S-boxes disponibles=20
z=4, rondas=2, estado=100 bits, S-boxes/ronda=20, S-boxes disponibles=40
z=4, rondas=3, estado=100 bits, S-boxes/ronda=20, S-boxes disponibles=60
z=8, rondas=1, estado=200 bits, S-boxes/ronda=40, S-boxes disponibles=40
z=8, rondas=2, estado=200 bits, S-boxes/ronda=40, S-boxes disponibles=80
z=8, rondas=3, estado=200 bits, S-boxes/ronda=40, S-boxes disponibles=120


## 5. Matriz de experimentos

La matriz experimental combina:

- dos tamaños de palabra: $(z=4)$ y $(z=8)$;
- tres números de rondas: 1, 2 y 3;
- el solver CBC como opción inicial.

La cantidad de S-boxes disponibles no representa el mínimo buscado. Solo indica el número total de instancias de $(\chi)$ existentes en cada configuración.

El modelo MILP determinará posteriormente cuántas de ellas deben estar activas como mínimo.

In [5]:
# ============================================================
# CONSTRUCCIÓN DE LA MATRIZ EXPERIMENTAL
# ============================================================

registros = []

for config in configuraciones:
    if config.rounds == 1:
        rango_intentos = "menos de 10"
    elif config.rounds == 2:
        rango_intentos = "menos de 20"
    else:
        rango_intentos = "menos de 30"

    registros.append(
        {
            "experimento": f"E{len(registros) + 1}",
            "z": config.z,
            "bits_estado": config.state_bits,
            "rondas": config.rounds,
            "intentos_referenciales": rango_intentos,
            "sboxes_por_ronda": config.sboxes_per_round,
            "sboxes_disponibles": config.total_sboxes,
            "solver": config.solver.upper(),
            "limite_tiempo_segundos": config.time_limit_seconds,
            "mip_gap": config.mip_gap,
        }
    )


df_experimentos = pd.DataFrame(registros)

df_experimentos

,experimento,z,bits_estado,rondas,intentos_referenciales,sboxes_por_ronda,sboxes_disponibles,solver,limite_tiempo_segundos,mip_gap
0,E1,4,100,1,menos de 10,20,20,CBC,300,0.0
1,E2,4,100,2,menos de 20,20,40,CBC,300,0.0
2,E3,4,100,3,menos de 30,20,60,CBC,300,0.0
3,E4,8,200,1,menos de 10,40,40,CBC,300,0.0
4,E5,8,200,2,menos de 20,40,80,CBC,300,0.0
5,E6,8,200,3,menos de 30,40,120,CBC,300,0.0


## 6. Validaciones automáticas de las dimensiones

Se comprueba que las configuraciones coincidan con la estructura de Keccak:

$[
\text{bits del estado}=25z
]$

$[
\text{S-boxes por ronda}=5z
]$

$[
\text{S-boxes disponibles}=5zR,
$]$

donde $(R)$ es el número de rondas.

In [6]:
# ============================================================
# VALIDACIONES AUTOMÁTICAS DE LAS DIMENSIONES
# ============================================================

# Validaciones para z = 4.
config_z4_r1 = ExperimentConfig(z=4, rounds=1)
config_z4_r3 = ExperimentConfig(z=4, rounds=3)

assert config_z4_r1.state_bits == 100
assert config_z4_r1.sboxes_per_round == 20
assert config_z4_r1.total_sboxes == 20
assert config_z4_r3.total_sboxes == 60

# Validaciones para z = 8.
config_z8_r1 = ExperimentConfig(z=8, rounds=1)
config_z8_r3 = ExperimentConfig(z=8, rounds=3)

assert config_z8_r1.state_bits == 200
assert config_z8_r1.sboxes_per_round == 40
assert config_z8_r1.total_sboxes == 40
assert config_z8_r3.total_sboxes == 120

# Validación general.
for config in configuraciones:
    assert config.state_bits == 25 * config.z
    assert config.sboxes_per_round == 5 * config.z
    assert config.total_sboxes == 5 * config.z * config.rounds


print("Todas las dimensiones experimentales fueron validadas correctamente.")

Todas las dimensiones experimentales fueron validadas correctamente.


## 7. Verificación del mapeo entre intentos y rondas

Para el desarrollo inicial se adopta la siguiente interpretación:

$[
r(I)=
\begin{cases}
1, & 1\leq I<10,\\
2, & 10\leq I<20,\\
3, & 20\leq I<30.
\end{cases}
]$

Esta relación representa la propuesta dinámica del algoritmo modificado: un mayor número de intentos implica la ejecución de un mayor número de rondas.

La función se implementa inicialmente como una regla experimental. Posteriormente deberá documentarse como supuesto metodológico o ajustarse según la precisión proporcionada por el docente.

In [7]:
# ============================================================
# MAPEO ENTRE NÚMERO DE INTENTOS Y NÚMERO DE RONDAS
# ============================================================

def rondas_desde_intentos(intentos: int) -> int:
    """
    Determina el número de rondas a partir del número de intentos.

    Reglas:
        1 a 9 intentos   -> 1 ronda
        10 a 19 intentos -> 2 rondas
        20 a 29 intentos -> 3 rondas

    Parameters
    ----------
    intentos:
        Número entero de intentos.

    Returns
    -------
    int
        Número de rondas asociadas.

    Raises
    ------
    ValueError
        Si el número de intentos se encuentra fuera del intervalo 1-29.
    """

    if not isinstance(intentos, int):
        raise TypeError("El número de intentos debe ser un valor entero.")

    if 1 <= intentos < 10:
        return 1

    if 10 <= intentos < 20:
        return 2

    if 20 <= intentos < 30:
        return 3

    raise ValueError(
        "El número de intentos debe encontrarse entre 1 y 29."
    )


casos_prueba = [1, 5, 9, 10, 15, 19, 20, 25, 29]

for intentos in casos_prueba:
    print(
        f"Intentos: {intentos:2d} -> "
        f"Rondas: {rondas_desde_intentos(intentos)}"
    )

Intentos:  1 -> Rondas: 1
Intentos:  5 -> Rondas: 1
Intentos:  9 -> Rondas: 1
Intentos: 10 -> Rondas: 2
Intentos: 15 -> Rondas: 2
Intentos: 19 -> Rondas: 2
Intentos: 20 -> Rondas: 3
Intentos: 25 -> Rondas: 3
Intentos: 29 -> Rondas: 3


In [8]:
# ============================================================
# PRUEBAS DEL MAPEO INTENTOS-RONDAS
# ============================================================

assert rondas_desde_intentos(1) == 1
assert rondas_desde_intentos(9) == 1

assert rondas_desde_intentos(10) == 2
assert rondas_desde_intentos(19) == 2

assert rondas_desde_intentos(20) == 3
assert rondas_desde_intentos(29) == 3

print("El mapeo entre intentos y rondas fue validado correctamente.")

El mapeo entre intentos y rondas fue validado correctamente.


## 8. Exportación de la matriz experimental

La matriz de configuraciones se guarda como archivo CSV para asegurar la trazabilidad de los experimentos.

El archivo generado será:

```text
results/tables/configuraciones_experimentales.csv

In [9]:


# ============================================================
# EXPORTACIÓN DE LA MATRIZ EXPERIMENTAL
# ============================================================

TABLES_DIR = PROJECT_ROOT / "results" / "tables"
TABLES_DIR.mkdir(parents=True, exist_ok=True)

ruta_salida = TABLES_DIR / "configuraciones_experimentales.csv"

df_experimentos.to_csv(
    ruta_salida,
    index=False,
    encoding="utf-8-sig",
)

print("=" * 70)
print("EXPORTACIÓN COMPLETADA")
print("=" * 70)
print(f"Archivo generado: {ruta_salida}")
print(f"Existe          : {ruta_salida.exists()}")
print("=" * 70)

EXPORTACIÓN COMPLETADA
Archivo generado: d:\Documentos\000. MSC\3er Ciclo\Cripto\PracticaCalificada\results\tables\configuraciones_experimentales.csv
Existe          : True


## 9. Resumen de la validación

En esta etapa se comprobó que:

1. el notebook reconoce correctamente la estructura del proyecto;
2. las dependencias principales están instaladas;
3. PuLP reconoce el solver CBC;
4. CBC resuelve correctamente un modelo MILP binario;
5. las dimensiones de Keccak para \(z=4\) y \(z=8\) son consistentes;
6. se definieron las seis configuraciones experimentales;
7. se implementó provisionalmente la relación entre intentos y rondas;
8. la matriz experimental fue exportada a un archivo reproducible.

La siguiente etapa consistirá en representar formalmente el estado:

$[
A[x,y,k],\qquad
x,y\in\{0,1,2,3,4\},\quad
k\in\{0,\ldots,z-1\},
]$

y programar las transformaciones lineales $(\rho)$ y $(\pi$).

In [10]:
# ============================================================
# RESUMEN FINAL DE LA ETAPA
# ============================================================

resumen_validacion = {
    "ruta_proyecto_valida": PROJECT_ROOT.exists(),
    "ruta_src_valida": SRC_DIR.exists(),
    "solver_cbc_disponible": "PULP_CBC_CMD" in available_solvers(),
    "prueba_milp_optima": estado_prueba == "Optimal",
    "numero_experimentos": len(df_experimentos),
    "archivo_csv_generado": ruta_salida.exists(),
}

df_resumen = pd.DataFrame(
    resumen_validacion.items(),
    columns=["validacion", "resultado"],
)

df_resumen

,validacion,resultado
0,ruta_proyecto_valida,True
1,ruta_src_valida,True
2,solver_cbc_disponible,True
3,prueba_milp_optima,True
4,numero_experimentos,6
5,archivo_csv_generado,True


In [11]:
# ============================================================
# CONTROL FINAL
# ============================================================

assert all(resumen_validacion.values()), (
    "Al menos una validación del entorno no fue superada."
)

print("=" * 70)
print("ENTORNO VALIDADO")
print("=" * 70)
print("El proyecto está listo para iniciar el modelado de Keccak.")
print("=" * 70)

ENTORNO VALIDADO
El proyecto está listo para iniciar el modelado de Keccak.
